# Belief propagation (sum-product, max-product) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Messages carry summarized evidence along a graph
Belief propagation turns a large sum or max into local messages. These examples run sum-product on a chain, normalize a belief, add evidence, compute max-product, and compare messages to brute force.

In [ ]:
# 1) sum-product message through a two-edge chain X-Y-Z
phiX=np.array([0.6,0.4]); psi=np.array([[0.9,0.1],[0.2,0.8]]); phiZ=np.array([0.3,0.7])
mXtoY=phiX; mZtoY=psi@phiZ; belY=normalize((psi.T@mXtoY)*mZtoY)
plt.figure(figsize=(6,3)); plt.bar(['Y=0','Y=1'],belY); plt.title('belief at Y from left and right messages')
assert abs(belY[0]-0.47222222222222215)<1e-12

In [ ]:
# 2) right evidence Z=1 sharpens the right-to-left message
eZ=np.array([0.,1.]); msg=psi@eZ; bel=normalize((psi.T@phiX)*msg)
plt.figure(figsize=(6,3)); plt.bar(['Y=0','Y=1'],bel); plt.title('hard evidence changes the message')
assert bel[1]>belY[1]

In [ ]:
# 3) max-product replaces sums with maxima for MAP
scoreY=(psi.T@phiX)*msg; max_msg=np.max(phiX[:,None]*psi,axis=0); map_bel=max_msg*msg
plt.figure(figsize=(6,3)); plt.bar(['Y=0','Y=1'],map_bel); plt.title('max-product scores MAP states')
assert int(np.argmax(map_bel))==1

In [ ]:
# 4) brute-force marginal agrees with sum-product
joint=np.zeros((2,2,2))
for x,y,z in itertools.product([0,1],repeat=3): joint[x,y,z]=phiX[x]*psi[x,y]*psi[y,z]*phiZ[z]
br=joint.sum(axis=(0,2)); br=br/br.sum()
plt.figure(figsize=(6,3)); plt.plot(br,'o-',label='brute'); plt.plot(belY,'x--',label='messages'); plt.legend(); plt.title('messages reproduce exact chain marginal')
assert np.allclose(br,belY)

In [ ]:
# 5) message normalization does not change beliefs
m1=10*phiX; m2=5*mZtoY; b=normalize((psi.T@m1)*m2)
plt.figure(figsize=(6,3)); plt.bar(['normalized messages','scaled messages'],[belY[0],b[0]]); plt.title('scaling cancels during normalization')
assert np.allclose(b,belY)